In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import h3

# Dynamically add the ai-service directory to Python path
NOTEBOOK_DIR = Path.cwd().resolve()
AI_SERVICE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR / "ai-service"
REPO_ROOT = AI_SERVICE_DIR.parent

if str(AI_SERVICE_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_DIR))

print(f"Repository Root : {REPO_ROOT}")
print(f"AI Service Root : {AI_SERVICE_DIR}")

Repository Root : C:\Users\Harsh\AeroSentinel
AI Service Root : C:\Users\Harsh\AeroSentinel\ai-service


In [2]:
# Resolve paths to sample data and geojson
air_csv = REPO_ROOT / "data" / "sample" / "air_quality.csv"
weather_csv = REPO_ROOT / "data" / "sample" / "weather.csv"
stations_geojson = REPO_ROOT / "data" / "geo" / "monitoring_stations.geojson"

# Load data into pandas dataframes
df_air = pd.read_csv(air_csv)
df_weather = pd.read_csv(weather_csv)

with open(stations_geojson, "r", encoding="utf-8") as f:
    stations_data = json.load(f)

print(f"Air Quality Data Shape : {df_air.shape}")
print(f"Weather Data Shape     : {df_weather.shape}")
print(f"Monitoring Stations    : {len(stations_data['features'])} stations found in GeoJSON")

Air Quality Data Shape : (10, 11)
Weather Data Shape     : (10, 7)
Monitoring Stations    : 2 stations found in GeoJSON


In [3]:
print("--- AIR QUALITY DATA SUMMARY ---")
print("Missing values per column:")
print(df_air.isnull().sum())
print(f"\nExact duplicate rows: {df_air.duplicated().sum()}")
print(f"Station/Timestamp duplicates: {df_air.duplicated(subset=['station_id', 'observed_at']).sum()}")

print("\n--- WEATHER DATA SUMMARY ---")
print("Missing values per column:")
print(df_weather.isnull().sum())
print(f"Timestamp duplicates: {df_weather.duplicated(subset=['observed_at']).sum()}")

--- AIR QUALITY DATA SUMMARY ---
Missing values per column:
observed_at    0
station_id     0
latitude       0
longitude      0
pm25           0
pm2_5          0
pm10           0
no2            0
so2            0
co             0
o3             0
dtype: int64

Exact duplicate rows: 0
Station/Timestamp duplicates: 0

--- WEATHER DATA SUMMARY ---
Missing values per column:
observed_at       0
station_id        0
temperature       0
humidity          0
wind_speed        0
wind_direction    0
pressure          0
dtype: int64
Timestamp duplicates: 5


In [4]:
print("--- PM2.5 Statistics (ug/m3) ---")
print(df_air["pm25"].describe())

print("\n--- Weather Numerical Statistics ---")
print(df_weather[["temperature", "humidity", "wind_speed", "wind_direction"]].describe())

# Check for physically impossible negative values
neg_pm25 = (df_air["pm25"] < 0).sum()
invalid_humidity = ((df_weather["humidity"] < 0) | (df_weather["humidity"] > 100)).sum()
print(f"\nNegative PM2.5 readings found       : {neg_pm25}")
print(f"Invalid Humidity readings (>100%/<0%): {invalid_humidity}")

--- PM2.5 Statistics (ug/m3) ---
count    10.000000
mean     53.700000
std       9.964269
min      39.800000
25%      45.900000
50%      53.550000
75%      60.725000
max      70.300000
Name: pm25, dtype: float64

--- Weather Numerical Statistics ---
       temperature   humidity  wind_speed  wind_direction
count    10.000000  10.000000   10.000000       10.000000
mean     26.380000  73.320000    1.600000      229.200000
std       0.994205   5.124191    0.466667       11.535453
min      25.000000  65.000000    1.000000      210.000000
25%      25.600000  69.500000    1.225000      221.250000
50%      26.300000  73.250000    1.550000      231.000000
75%      27.100000  77.625000    1.950000      238.750000
max      28.000000  80.200000    2.400000      245.000000

Negative PM2.5 readings found       : 0
Invalid Humidity readings (>100%/<0%): 0


In [5]:
# Helper for H3 compatibility
def get_h3_res8(lat, lon):
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, 8)
    return h3.geo_to_h3(lat, lon, 8)

# Map station coordinates and H3 cell
station_coords = {}
for feat in stations_data["features"]:
    sid = feat["properties"]["station_id"]
    sname = feat["properties"]["station_name"]
    lon, lat = feat["geometry"]["coordinates"]
    cell = get_h3_res8(lat, lon)
    station_coords[sid] = {"lat": lat, "lon": lon, "name": sname, "h3_res8": cell}
    print(f"Station {sid:10s} ({sname:15s}) -> Lat: {lat:.4f}, Lon: {lon:.4f} -> H3: {cell}")

# Attach coordinates and H3 cell ID to the air quality dataframe
df_air["latitude"] = df_air["station_id"].map(lambda x: station_coords[x]["lat"])
df_air["longitude"] = df_air["station_id"].map(lambda x: station_coords[x]["lon"])
df_air["h3_cell_id"] = df_air["station_id"].map(lambda x: station_coords[x]["h3_res8"])

df_air.head()

Station ST_001     (Bandra Kurla Complex) -> Lat: 19.0760, Lon: 72.8777 -> H3: 88608b0b61fffff
Station ST_002     (Andheri East   ) -> Lat: 19.1136, Lon: 72.8697 -> H3: 88608b5495fffff


,observed_at,station_id,latitude,longitude,pm25,pm2_5,pm10,no2,so2,co,o3,h3_cell_id
0,2026-03-01 00:00:00,ST_001,19.076,72.8777,45.2,45.2,85.1,28.4,12.1,0.8,18.5,88608b0b61fffff
1,2026-03-01 01:00:00,ST_001,19.076,72.8777,48.0,48.0,91.3,30.1,13.0,0.9,16.2,88608b0b61fffff
2,2026-03-01 02:00:00,ST_001,19.076,72.8777,42.5,42.5,80.0,25.3,11.5,0.7,19.0,88608b0b61fffff
3,2026-03-01 03:00:00,ST_001,19.076,72.8777,39.8,39.8,74.6,22.0,10.2,0.6,21.3,88608b0b61fffff
4,2026-03-01 04:00:00,ST_001,19.076,72.8777,52.1,52.1,98.4,35.7,14.8,1.1,14.0,88608b0b61fffff
